# EPUB Audiobook - remote chunk synthesis

This notebook synthesizes the text chunks exported by the EPUB Audiobook App
for **one patch**, using the same `VoxCPM2` model the app uses locally, and
writes the resulting `chunk_NNN.wav` files into an **`output/`** subfolder inside
the exported folder, so the app's **Import from Drive** button can pick them up.

> **Enable a GPU first.** This model runs on CUDA. In Colab: **Runtime > Change runtime type > GPU (T4)**, then restart the session. Pick **GPU, not TPU** - VoxCPM cannot use a TPU and will silently fall back to CPU (extremely slow). Cell 6 checks this for you.


It reads everything it needs from `manifest.json`, which was exported
alongside this notebook - you should not need to type any patch info by hand.

## Google Colab (recommended)
The app already uploaded this folder into **your own Google Drive** (the
account you connected). Just run the cells top to bottom: cell 3 mounts your
Drive, and the folder is a normal filesystem path from then on - no Google API
calls needed in this notebook at all.

## Kaggle
Kaggle has no native Google Drive mount. The practical flow there:
1. In the app, use **Download package locally** (instead of, or in addition
   to, exporting to Drive) to get a `.zip` of this same folder.
2. Upload that zip as a Kaggle Dataset and attach it to this notebook.
3. Skip the Drive-mount cell below, and instead set `FOLDER_PATH` to your
   Kaggle dataset input path (e.g. `/kaggle/input/<dataset-name>`).
4. `/kaggle/input` is read-only, so the notebook writes the generated
   `chunk_NNN.wav` files to `/kaggle/working/output` instead (not inside the
   attached dataset). Download them from Kaggle's **Output** pane/file
   browser (`/kaggle/working/output`) when the run finishes.
5. Upload those `.wav` files (or a .zip of them) directly in the app's chunk
   page - no Drive connection required - or drop them into an `output/`
   subfolder inside the matching Drive folder yourself. Either way works: the
   app looks for `output/` first and falls back to the folder's top level for
   older exports.

In [ ]:
# Cell 1: install dependencies
!pip install -q voxcpm soundfile numpy

In [ ]:
# Cell 2: (optional) Hugging Face token - avoids the "unauthenticated requests" rate
# limit warning/slow downloads when fetching the model. Get a free token at
# https://huggingface.co/settings/tokens
#
# Recommended: store it as a secret instead of pasting it in plain text here -
# Colab: left sidebar > key icon > add secret named HF_TOKEN.
# Kaggle: Add-ons > Secrets > add secret named HF_TOKEN.
# If no secret is found, you'll get a hidden prompt to paste it manually (or just
# press Enter to skip and continue unauthenticated).
# The app replaces __HF_TOKEN__ below with the token from its own settings on export;
# leave the app's HF_TOKEN setting empty to keep this as a placeholder (secrets/prompt).
HF_TOKEN = "__HF_TOKEN__"

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or ""
except Exception:
    pass

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN") or ""
    except Exception:
        pass

if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("Hugging Face token (leave blank to skip): ")

if HF_TOKEN:
    import os
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub.")
else:
    print("No HF token set - continuing unauthenticated (may hit rate limits).")

In [ ]:
# Cell 3: Google Colab only - mount your Drive. The exported folder is located
# automatically by patch id, so you do NOT need to paste the folder name by hand.
# On Kaggle this cell automatically does nothing - set FOLDER_PATH in Cell 4
# instead (Kaggle ships the google.colab package, so importing it succeeding
# does NOT mean we're on Colab; drive.mount() there raises NotImplementedError).
import glob, json, os

IS_KAGGLE = os.path.isdir("/kaggle")

if IS_KAGGLE:
    print("Kaggle detected - skipping the Colab Drive mount (set FOLDER_PATH in Cell 4).")
else:
    from google.colab import drive

    drive.mount('/content/drive')

    PATCH_ID = __PATCH_ID__  # injected by the app when this notebook was exported
    EXPORTS_ROOT = "/content/drive/MyDrive/EPUB Audiobook Exports"
    DEFAULT_FOLDER = os.path.join(EXPORTS_ROOT, "__DEFAULT_FOLDER_NAME__")

    # Scan every export folder and match the one whose manifest.json has our patch id.
    FOLDER_PATH = None
    for d in sorted(glob.glob(os.path.join(EXPORTS_ROOT, "*")), reverse=True):
        manifest_path = os.path.join(d, "manifest.json")
        if not os.path.isfile(manifest_path):
            continue
        try:
            with open(manifest_path, encoding="utf-8") as f:
                if json.load(f).get("patch_id") == PATCH_ID:
                    FOLDER_PATH = d
                    break
        except Exception:
            pass

    if FOLDER_PATH is None:
        FOLDER_PATH = DEFAULT_FOLDER  # fall back to the exact name the app used at export time

    print("Using folder:", FOLDER_PATH)
    assert os.path.isdir(FOLDER_PATH), (
        f"Folder not found: {FOLDER_PATH}\n"
        "Make sure the export was uploaded to this Google account's Drive, "
        "or set FOLDER_PATH manually."
    )

In [ ]:
# Cell 4: Kaggle only - point at the attached dataset instead of Drive.
# This is a read-only mount - Cell 8 automatically writes output .wav files
# to /kaggle/working/output instead, so you don't need to change anything else.
# Use the EXACT zip filename (without .zip) as the dataset name when uploading the
# package to Kaggle - it already follows the same naming convention as the Drive
# folder, so you don't have to retype the format. e.g. if you downloaded
# "My Book - patch 0 - 20260701-143022.zip", create the Kaggle dataset with the
# name "My Book - patch 0 - 20260701-143022" and point FOLDER_PATH at it.
# FOLDER_PATH = "/kaggle/input/datasets/yukihuy9999/My Book - patch 0 - 20260701-143022"

In [ ]:
# Cell 5: load the manifest and the voice reference clip. The clip is REQUIRED:
# without it VoxCPM picks a different random voice per chunk/session, so the
# merged audio would not sound consistent.
import json
import os

with open(os.path.join(FOLDER_PATH, "manifest.json"), "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"Patch {manifest['patch_id']} of book '{manifest['book_title']}'")
print(f"{manifest['chunk_count']} chunks, max_chars={manifest['max_chars']}")

reference_wav_path = None
prompt_text = None
if manifest.get("reference_wav"):
    reference_wav_path = os.path.join(FOLDER_PATH, manifest["reference_wav"])
    prompt_text = manifest.get("reference_transcript") or None

if not reference_wav_path or not os.path.exists(reference_wav_path):
    raise RuntimeError(
        "Voice reference clip not found in this export - it is required so every "
        "chunk is synthesized with the same voice. In the app, upload a voice "
        "reference clip for this book, then re-export the patch."
    )
print(f"Using cloned voice reference: {reference_wav_path}")

In [ ]:
# Cell 6: check you're actually on a GPU (VoxCPM is far too slow on CPU).
# If this stops with an error, go to Runtime > Change runtime type > Hardware
# accelerator > GPU (T4), then Runtime > Restart session and run again.
# NOTE: pick GPU, NOT TPU - VoxCPM uses CUDA and cannot run on a TPU.
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU detected - VoxCPM would run on CPU and be extremely slow. "
        "Colab: Runtime > Change runtime type > GPU (T4), then Restart session. "
        "Choose GPU, not TPU. On Kaggle: enable a GPU accelerator in the sidebar."
    )

In [ ]:
# Cell 7: load the model (same model id the app uses locally)
from voxcpm import VoxCPM

model = VoxCPM.from_pretrained(manifest.get("voxcpm_model_id", "openbmb/VoxCPM2"), load_denoiser=False)

In [ ]:
# Cell 8: synthesize the chunks and write chunk_NNN.wav into an 'output' subfolder
# (keeps generated audio separate from the exported chunk_NNN.txt/manifest files).
import soundfile as sf

# --- config: which chunks to run in THIS session ---
# Use these to split work across accounts/runs or to skip early chunks.
START_INDEX = 0      # first chunk to synthesize (e.g. 1 to start at chunk_001)
END_INDEX = None     # last chunk to synthesize, inclusive; None = go to the end
SKIP_EXISTING = True # skip chunks that already have a .wav (safe to resume)

# /kaggle/input is a read-only mount - writing there raises a generic libsndfile
# "System error" (not a permissions message), so on Kaggle write to /kaggle/working
# instead. On Colab, FOLDER_PATH is the mounted (writable) Drive folder, so writing
# straight into an 'output' subfolder there is fine.
if FOLDER_PATH.startswith("/kaggle/input"):
    OUTPUT_DIR = "/kaggle/working/output"
else:
    OUTPUT_DIR = os.path.join(FOLDER_PATH, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Writing output .wav files to:", OUTPUT_DIR)

sample_rate = model.tts_model.sample_rate

for chunk_filename in manifest["chunks"]:
    index = chunk_filename.split("_")[1].split(".")[0]  # chunk_000.txt -> 000
    n = int(index)
    if n < START_INDEX:
        continue
    if END_INDEX is not None and n > END_INDEX:
        break

    out_path = os.path.join(OUTPUT_DIR, f"chunk_{index}.wav")
    if SKIP_EXISTING:
        # Also check in FOLDER_PATH/output/ for files uploaded back to the dataset
        dataset_wav = os.path.join(FOLDER_PATH, "output", f"chunk_{index}.wav")
        if os.path.exists(out_path) or os.path.exists(dataset_wav):
            print(f"skip {chunk_filename} (already synthesized)")
            continue

    with open(os.path.join(FOLDER_PATH, chunk_filename), "r", encoding="utf-8") as f:
        text = f.read()

    kwargs = {}
    if reference_wav_path:
        kwargs["reference_wav_path"] = reference_wav_path
        if prompt_text:
            kwargs["prompt_wav_path"] = reference_wav_path
            kwargs["prompt_text"] = prompt_text

    audio = model.generate(text=text, cfg_value=2.0, inference_timesteps=10, **kwargs)
    sf.write(out_path, audio, sample_rate)
    print(f"wrote {out_path}")

print("Done. Go back to the app and import the results.")

In [ ]:
import soundfile as sf
import numpy as np
import os

all_audio_data = []

# Sort chunks by index to ensure correct order
sorted_chunk_files = sorted(manifest["chunks"], key=lambda x: int(x.split('_')[1].split('.')[0]))

print(f"Attempting to merge {len(sorted_chunk_files)} audio chunks.")

for i, chunk_filename in enumerate(sorted_chunk_files):
    index = chunk_filename.split("_")[1].split(".")[0]
    wav_path = os.path.join(FOLDER_PATH, "output", f"chunk_{index}.wav")
    
    if os.path.exists(wav_path):
        try:
            audio, sr = sf.read(wav_path)
            all_audio_data.append(audio)
            print(f"({i+1}/{len(sorted_chunk_files)}) Successfully read: {wav_path}")
        except Exception as e:
            print(f"({i+1}/{len(sorted_chunk_files)}) Error reading {wav_path}: {e}. Skipping.")
    else:
        print(f"({i+1}/{len(sorted_chunk_files)}) Warning: {wav_path} not found. Skipping.")

print(f"Found {len(all_audio_data)} audio files to merge.")

if all_audio_data:
    combined_audio = np.concatenate(all_audio_data)
    output_combined_path = os.path.join(FOLDER_PATH, "output", "combined_audio.wav")
    sf.write(output_combined_path, combined_audio, sample_rate)
    print(f"Successfully merged all chunks into: {output_combined_path}")
else:
    print("No audio chunks found to merge. The combined_audio.wav file was not created.")

In [ ]:
# Cell 10: Check FFmpeg is available.
import subprocess

result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
if result.returncode == 0:
    print("FFmpeg OK:", result.stdout.splitlines()[0])
else:
    raise RuntimeError("FFmpeg not found. Install with: !apt-get install -y ffmpeg")

In [ ]:
# Cell 11: Render MP4 from merged audio + background image + optional music + marquee.
# Resume-safe: skips if result.mp4 already exists.
import json
import os
import subprocess

video_config = manifest.get("video_config", {})
music_rel = video_config.get("music_file")
music_volume = video_config.get("music_volume", 0.15)
fps = video_config.get("fps", 30)
resolution = video_config.get("resolution", "1920x1080")

patch_label = manifest.get("patch_name", "patch")
patch_index = manifest.get("patch_id", 0)
result_mp4 = os.path.join(FOLDER_PATH, "output", "result.mp4")
result_wav = os.path.join(FOLDER_PATH, "output", "combined_audio.wav")
bg_abs = os.path.join(FOLDER_PATH, manifest["background_image"]) if manifest.get("background_image") else None
music_abs = os.path.join(FOLDER_PATH, music_rel) if music_rel else None

# Marquee band: <patch_id>.marquee.png + <patch_id>.marquee.json bundled alongside background
marquee_png = os.path.join(FOLDER_PATH, f"{patch_index}.marquee.png")
marquee_json = os.path.join(FOLDER_PATH, f"{patch_index}.marquee.json")
marquee_meta = None
if os.path.exists(marquee_png) and os.path.exists(marquee_json):
    with open(marquee_json, encoding="utf-8") as f:
        marquee_meta = json.load(f)

if os.path.exists(result_mp4):
    print(f"MP4 already exists: {result_mp4} \u2014 skipping")
elif not os.path.exists(result_wav):
    print("Merged WAV not found \u2014 run Cell 8 (merge) first")
elif not bg_abs or not os.path.exists(bg_abs):
    print("Background image not found \u2014 cannot render video")
else:
    if music_abs and not os.path.exists(music_abs):
        print(f"Music file not found at {music_abs} \u2014 rendering without music")
        music_abs = None

    os.makedirs(os.path.dirname(result_mp4), exist_ok=True)
    w_h = resolution.replace("x", ":")

    # Build inputs
    inputs = ["-loop", "1", "-i", bg_abs, "-i", result_wav]
    next_idx = 2
    music_idx = None
    marquee_idx = None
    if music_abs:
        inputs += ["-stream_loop", "-1", "-i", music_abs]
        music_idx = next_idx; next_idx += 1
    if marquee_meta:
        inputs += ["-loop", "1", "-i", marquee_png]
        marquee_idx = next_idx; next_idx += 1

    base_vf = f"scale={w_h}:force_original_aspect_ratio=decrease,pad={w_h}:(ow-iw)/2:(oh-ih)/2"
    audio_chains = []
    if music_idx is not None:
        audio_chains.append(f"[{music_idx}:a]volume={music_volume}[music]")
        audio_chains.append("[1:a][music]amix=inputs=2:duration=first:normalize=0[aout]")
        audio_map = "[aout]"
    else:
        audio_map = "1:a"

    if marquee_idx is not None and marquee_meta:
        band_h = marquee_meta["marquee_height"]
        speed = marquee_meta["speed_px_per_sec"]
        scroll_unit = max(1, marquee_meta["scroll_unit_px"])
        w = int(resolution.split("x")[0])
        band_vf = f"crop={w}:{band_h}:x='(t*{speed})%{scroll_unit}':y=0"
        chains = audio_chains + [
            f"[0:v]{base_vf}[bg]",
            f"[{marquee_idx}:v]{band_vf}[band]",
            "[bg][band]overlay=0:0[outv]",
        ]
        cmd = ["ffmpeg", "-y", *inputs,
               "-filter_complex", ";".join(chains),
               "-map", "[outv]", "-map", audio_map,
               "-c:v", "libx264", "-tune", "stillimage",
               "-r", str(fps), "-c:a", "aac", "-b:a", "192k",
               "-pix_fmt", "yuv420p", "-crf", "23", "-shortest", result_mp4]
    elif music_idx is not None:
        chains = audio_chains + [f"[0:v]{base_vf}"]
        cmd = ["ffmpeg", "-y", *inputs,
               "-filter_complex", ";".join(chains),
               "-map", "0:v", "-map", audio_map,
               "-c:v", "libx264", "-tune", "stillimage",
               "-r", str(fps), "-c:a", "aac", "-b:a", "192k",
               "-pix_fmt", "yuv420p", "-crf", "23", "-shortest", result_mp4]
    else:
        cmd = ["ffmpeg", "-y", *inputs,
               "-vf", base_vf,
               "-c:v", "libx264", "-tune", "stillimage",
               "-r", str(fps), "-c:a", "aac", "-b:a", "192k",
               "-pix_fmt", "yuv420p", "-crf", "23", "-shortest", result_mp4]

    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print("FFmpeg error:", proc.stderr[-1000:])
    else:
        print(f"Rendered: {result_mp4}")

In [ ]:
# Cell 12: Upload rendered MP4 to YouTube.
import json
import os

YOUTUBE_CREDS = None
try:
    from google.colab import userdata
    YOUTUBE_CREDS = json.loads(userdata.get("YOUTUBE_CREDS") or "{}")
except Exception:
    pass

if not YOUTUBE_CREDS:
    try:
        from kaggle_secrets import UserSecretsClient
        YOUTUBE_CREDS = json.loads(UserSecretsClient().get_secret("YOUTUBE_CREDS") or "{}")
    except Exception:
        pass

result_mp4 = os.path.join(FOLDER_PATH, "output", "result.mp4")
id_file = result_mp4 + ".youtube_id"

if not YOUTUBE_CREDS or not YOUTUBE_CREDS.get("refresh_token"):
    print("YOUTUBE_CREDS not found. Get it from /youtube in the app -> 'Copy YouTube credentials'.")
elif os.path.exists(id_file):
    vid_id = open(id_file).read().strip()
    print(f"Already uploaded: https://youtube.com/watch?v={vid_id}")
elif not os.path.exists(result_mp4):
    print("MP4 not found — run Cell 11 first")
else:
    from google.oauth2.credentials import Credentials
    from google.auth.transport.requests import Request
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload

    creds = Credentials(
        token=None,
        refresh_token=YOUTUBE_CREDS["refresh_token"],
        token_uri="https://oauth2.googleapis.com/token",
        client_id=YOUTUBE_CREDS["client_id"],
        client_secret=YOUTUBE_CREDS["client_secret"],
        scopes=["https://www.googleapis.com/auth/youtube.upload"],
    )
    creds.refresh(Request())
    yt = build("youtube", "v3", credentials=creds)

    title = f"{manifest['book_title']} - {manifest['patch_name']}"
    privacy = video_config.get("youtube_privacy", "private")
    body = {
        "snippet": {"title": title[:100], "description": manifest["book_title"], "categoryId": "26"},
        "status": {"privacyStatus": privacy},
    }
    media = MediaFileUpload(result_mp4, mimetype="video/mp4", resumable=True, chunksize=10*1024*1024)
    req = yt.videos().insert(part="snippet,status", body=body, media_body=media)
    response = None
    while response is None:
        status, response = req.next_chunk()
        if status:
            print(f"  {int(status.progress()*100)}%")
    vid_id = response["id"]
    with open(id_file, "w") as f:
        f.write(vid_id)
    print(f"Uploaded: https://youtube.com/watch?v={vid_id}")